# 문맥 보존형 뉴스-KOSIS 파이프라인 smoke test

숫자가 포함된 기사 10개를 고정 seed로 선택해 KSS, 중첩 chunk, 넓은 claim span, 조기 BGE, HCX measurement, 3단계 게이트, KOSIS 실제값 검증까지 실행합니다.

Colab 보안 비밀에 `CLOVA_API_KEY`, `KOSIS_API_KEY`를 등록하세요. 런타임은 GPU를 선택합니다.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')

import csv
import json
import os
import random
import shutil
import subprocess
import sys
from collections import Counter
from pathlib import Path

REPO_URL = 'https://github.com/rnwjdgus03/NLP_05-Team-Project-3.git'
BRANCH = 'codex/repro-baseline-20260727'
REPO_DIR = Path('/content/NLP_05-Team-Project-3')
DRIVE_ROOT = Path('/content/drive/MyDrive/NLP_05-Team-Project-3')
INPUT_DIR = DRIVE_ROOT / 'inputs'
RUN_DIR = DRIVE_ROOT / 'runs' / 'contextual_smoke_context_v2_10'
INDEX_DIR = DRIVE_ROOT / 'indexes' / 'kosis_bge_m3'
ARTICLE_CSV = INPUT_DIR / 'news_articles.csv'
SMOKE_ARTICLES = RUN_DIR / '00_smoke_articles_10.csv'
EARLY_META_CANDIDATES = [
    DRIVE_ROOT / 'runs' / 'early_bge_rag_5000' / 'early_bge_meta_index.csv',
    DRIVE_ROOT / 'runs' / 'early_bge_rag' / 'early_bge_meta_index.csv',
]
INPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('run:', RUN_DIR)

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'requests>=2.31,<3', 'python-dotenv>=1.0,<2', 'kss>=6,<7',
    'numpy>=1.26,<3', 'sentence-transformers>=3.4,<6', 'transformers>=4.45,<6'
], check=True)
os.chdir(REPO_DIR)
print('repo:', REPO_DIR)
print('branch:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True).strip())

## 원문 기사 CSV 준비

`MyDrive/NLP_05-Team-Project-3/inputs/news_articles.csv`가 없으면 업로드 창이 열립니다. 문장 CSV가 아니라 기사 본문 컬럼이 있는 원문 CSV를 선택하세요.

In [ ]:
if not ARTICLE_CSV.exists():
    print('기사 원문 CSV를 선택하세요.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('기사 원문 CSV 한 개만 업로드하세요.')
    uploaded_name = next(iter(uploaded))
    shutil.copy2(uploaded_name, ARTICLE_CSV)
print('article CSV:', ARTICLE_CSV, ARTICLE_CSV.stat().st_size)

In [ ]:
from preprocess_news import read_articles, resolve_columns

articles, fieldnames, encoding = read_articles(ARTICLE_CSV, 'auto')
columns = resolve_columns(fieldnames, {})
body_col = columns['body']
numeric_articles = [row for row in articles if any(ch.isdigit() for ch in str(row.get(body_col, '') or ''))]
if len(numeric_articles) < 10:
    raise RuntimeError(f'숫자가 포함된 기사가 부족합니다: {len(numeric_articles)}')
sample = random.Random(20260729).sample(numeric_articles, 10)
with SMOKE_ARTICLES.open('w', encoding='utf-8-sig', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(sample)
print(f'articles={len(articles):,} numeric={len(numeric_articles):,} smoke=10 encoding={encoding}')
print('smoke input:', SMOKE_ARTICLES)

In [ ]:
for key in ('CLOVA_API_KEY', 'KOSIS_API_KEY'):
    if not os.environ.get(key):
        os.environ[key] = userdata.get(key) or ''
    if not os.environ.get(key):
        raise RuntimeError(f'Colab 보안 비밀에 {key}를 등록하세요.')

for path, label in [
    (INDEX_DIR / 'manifest.json', 'BGE manifest'),
    (INDEX_DIR / 'embeddings.npy', 'BGE embeddings'),
    (REPO_DIR / 'kosis_table_summary.csv', 'KOSIS table index'),
]:
    if not path.exists():
        raise FileNotFoundError(f'{label}가 없습니다: {path}')
EARLY_META = next((path for path in EARLY_META_CANDIDATES if path.exists()), None)
print('inputs and secrets: ready')
print('early meta:', EARLY_META or '없음 - 통계표명/분류만 HCX에 제공')

In [ ]:
command = [
    sys.executable, '-u', str(REPO_DIR / 'run_contextual_news_kosis_pipeline.py'),
    '--articles', str(SMOKE_ARTICLES),
    '--table-index', str(REPO_DIR / 'kosis_table_summary.csv'),
    '--semantic-index', str(INDEX_DIR),
    '--out-dir', str(RUN_DIR),
    '--device', 'cuda',
    '--verify',
]
if EARLY_META:
    command.extend(['--early-meta-index', str(EARLY_META)])
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
def read_rows(path):
    if not path.exists():
        return []
    with path.open(encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))

paths = {
    'sentences': RUN_DIR / '01_sentences.csv',
    'chunks': RUN_DIR / '02_chunks.csv',
    'claim spans': RUN_DIR / '03_claim_spans.csv',
    'claim contexts': RUN_DIR / '03_claim_contexts.csv',
    'early candidates': RUN_DIR / '04_early_bge_candidates_top20.csv',
    'measurements': RUN_DIR / '05_hcx_measurements.csv',
    'READY': RUN_DIR / '06_mapping_ready.csv',
    'ENRICH': RUN_DIR / '06_mapping_enrich.csv',
    'REJECT': RUN_DIR / '06_mapping_reject.csv',
    'validated mappings': RUN_DIR / '07_mapping' / '05_hcx_measurements_kosis_validated_mappings.csv',
    'verified': RUN_DIR / '07_mapping' / '05_hcx_measurements_kosis_verified.csv',
}
for label, path in paths.items():
    rows = read_rows(path)
    claim_count = len({row.get('claim_id') for row in rows if row.get('claim_id')})
    print(f'{label:20s}: rows={len(rows):4d}, claims={claim_count:3d}, file={path.name}')

In [ ]:
enrich_rows = read_rows(paths['ENRICH'])
reject_rows = read_rows(paths['REJECT'])
verified_rows = read_rows(paths['verified'])

print('ENRICH actions')
print(Counter(row.get('enrichment_actions', '-') for row in enrich_rows))
print('\nREJECT reasons')
print(Counter(row.get('mapping_exclusion_code', '-') for row in reject_rows))
print('\nverdicts')
verdict_field = next((field for field in ('verdict', 'verification_result', 'kosis_verdict') if verified_rows and field in verified_rows[0]), None)
print(Counter(row.get(verdict_field, '-') for row in verified_rows) if verdict_field else 'verified 파일의 verdict 컬럼을 확인하세요.')
print('\n최종 확인 파일:', paths['verified'])